In [1]:
import pandas as pd

rfm = pd.read_csv("../data/rfm_churn.csv")
rfm.head()

,customer_unique_id,recency,frequency,monetary,churned
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,129.90,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,18.90,0
2,0000f46a3911fa3c0805444483337064,537,1,69.00,1
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,25.99,1
4,0004aac84e0df4da2b147fca70cf8255,288,1,180.00,1


In [8]:

X = rfm[["frequency", "monetary"]]
y = rfm["churned"]

print(X.shape, y.shape)

(93358, 2) (93358,)


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train size:", X_train.shape[0])
print("Test size:", X_test.shape[0])

Train size: 74686
Test size: 18672


In [10]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight="balanced")
model.fit(X_train, y_train)

print("Model trained")

Model trained


In [11]:
from sklearn.metrics import classification_report, roc_auc_score

preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, preds))
print("ROC-AUC:", roc_auc_score(y_test, probs))

              precision    recall  f1-score   support

           0       0.41      0.19      0.26      7621
           1       0.59      0.81      0.68     11051

    accuracy                           0.56     18672
   macro avg       0.50      0.50      0.47     18672
weighted avg       0.52      0.56      0.51     18672

ROC-AUC: 0.5100046638747853


In [12]:
import pandas as pd

coefficients = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", ascending=False)

coefficients

,feature,coefficient
1,monetary,-0.000093
0,frequency,-0.129562


## Baseline Model
- Initial attempt included `recency` as a feature — resulted in ROC-AUC of 1.0,
  which revealed data leakage (churn is defined AS recency > 180 days, so including
  it makes the "prediction" circular).
- Fixed by removing recency, predicting churn using only frequency and monetary.
- ROC-AUC after fix: 0.5100046638747853